# ⚽ TAJ — Football Tactical Intelligence · Colab Edition
**Open-data, evidence-led spatial analysis** — قابل اجرا روی Google Colab بدون GPU.

> داده‌های Demo کاملاً **مصنوعی** هستند. ادعاهای تاکتیکی فقط زمانی معتبرتر می‌شوند که روی Tracking واقعی و با شواهد مستقل بررسی شوند. این نوت‌بوک هیچ پیش‌بینی نتیجه مسابقه ارائه نمی‌کند.

[Repository](https://github.com/prestigegitserp/Taj_soccer_recommender) · [Static Visual Lab](https://prestigegitserp.github.io/Taj_soccer_recommender/) (پس از فعال‌سازی Pages)

**مراحل:** نصب → Demo → Heatmap → گزارش و پیشنهاد → IDSSE واقعی → SkillCorner واقعی.

## 1 · Install dependencies and fetch your repository
این سلول را یک بار اجرا کن. نصب به اینترنت Colab نیاز دارد.

In [ ]:
import os, sys, subprocess
os.chdir("/content")
if not os.path.isdir("/content/Taj_soccer_recommender"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/prestigegitserp/Taj_soccer_recommender.git"],check=True)
os.chdir("/content/Taj_soccer_recommender")
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[all]","ipywidgets"],check=True)
print("✅ Workspace:",os.getcwd())

## 2 · Smoke-test with **synthetic** data
این داده برای آزمایش کد و رابط کاربری است و نباید به‌عنوان نتیجه بازی واقعی گزارش شود.

In [ ]:
from taj.analysis import create_demo, build_report, export_report
from taj.plotting import pitch_figure
from taj.schema import at_frame,validate_tracking
from IPython.display import display, Markdown
import pandas as pd
frames, ball, report = create_demo("data/demo")
display(Markdown("**Rows:** "+str(len(frames))+" · **Synthetic:** "+str(report["synthetic"])))
display(report["quality"])

## 3 · Interactive tactical pitch
اسلایدر را حرکت بده و تیک Heatmap را تغییر بده؛ مدل شاخص دسترسی فضایی است، نه احتمال کالیبره‌شده.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
indices = frames[["period","frame_id","t_s"]].drop_duplicates().reset_index(drop=True)
slider=widgets.IntSlider(value=len(indices)//2,min=0,max=len(indices)-1,step=1,
                         description="Frame",continuous_update=False,layout=widgets.Layout(width="70%"))
heat=widgets.Checkbox(value=True,description="Pitch access")
out=widgets.Output()
def render(_=None):
    with out:
        clear_output(wait=True)
        k=indices.iloc[slider.value]
        f=at_frame(frames,int(k.period),str(k.frame_id))
        b=ball[ball.frame_id==str(k.frame_id)]
        display(pitch_figure(f,b,heatmap=heat.value,
                    title="SYNTHETIC TEST · %.1f s"%k.t_s))
slider.observe(render,names="value")
heat.observe(render,names="value")
display(widgets.VBox([widgets.HBox([slider,heat]),out]))
render()

## 4 · Auditable tactical findings
پیشنهاد فقط زمانی ساخته می‌شود که دست‌کم چند پنجره زمانی مستقل از فرضیه پشتیبانی کنند.

In [ ]:
report=build_report(frames,ball,defending="Home",goal=1,stride=3,min_windows=2)
display(Markdown("### Hypotheses ("+str(len(report["findings"]))+")"))
for f in report["findings"]:
    display(Markdown("**"+f["statement_fa"]+"** · "+str(f["n_independent_windows"])+" windows"))
display(Markdown("### Suggestions ("+str(len(report["recommendations"]))+")"))
for r in report["recommendations"]:
    display(Markdown("- **"+r["title_fa"]+"** · "+r["risk_fa"]))
p=export_report(report,"outputs/demo_report.json")
print("Report saved:",p)

## 5 · Real IDSSE data
**اجرا اختیاری**: متغیر RUN_IDSSE را True کن؛ دانلود فایل‌های واقعی ممکن است چند دقیقه و حافظه بیشتر لازم داشته باشد. ID های مجاز در `taj.ingest.IDSSE_IDS` قرار دارند.

In [ ]:
RUN_IDSSE = False
if RUN_IDSSE:
    from taj.ingest import load_idsse,write_bundle
    tracking, real_ball, real_events = load_idsse(
        match_id="J03WMX",limit=1200,sample_rate=2,with_events=False)
    stats = write_bundle(tracking,real_ball,real_events,"data/idsse_J03WMX")
    print(stats)
    r=build_report(tracking,real_ball,real_events,stride=10,min_windows=4)
    print("Real-data hypotheses:",len(r["findings"]))
    display(pitch_figure(at_frame(tracking,int(tracking.period.iloc[0]),
                           str(tracking.frame_id.iloc[0])),title="IDSSE / real tracking"))
else:
    print("IDSSE download skipped. Set RUN_IDSSE = True when ready.")

## 6 · Real SkillCorner via Git LFS
**اجرا اختیاری**: Match 1925299 شامل داده‌های حجیم Git LFS است. در Colab فایل‌های واقعی لازم‌اند؛ Pointer صدبایتی فایل داده نیست. `is_detected` جداگانه ذخیره می‌شود.

In [ ]:
RUN_SKILLCORNER = False
if RUN_SKILLCORNER:
    import shutil
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","git-lfs"],check=True)
    subprocess.run(["git","lfs","install"],check=True)
    if not os.path.isdir("external/opendata"):
        env={**os.environ,"GIT_LFS_SKIP_SMUDGE":"1"}
        subprocess.run(["git","clone","--depth","1","https://github.com/SkillCorner/opendata.git",
                       "external/opendata"],env=env,check=True)
    subprocess.run(["git","lfs","pull","-I",
                    "data/matches/1925299/1925299_tracking_extrapolated.jsonl"],
                   cwd="external/opendata",check=True)
    from taj.ingest import load_skillcorner_local, write_bundle
    sc,bsc,esc=load_skillcorner_local("1925299","external/opendata",
                                     limit=2000,stride=10)
    print(write_bundle(sc,bsc,esc,"data/skillcorner_1925299"))
else:
    print("SkillCorner LFS download skipped. Set RUN_SKILLCORNER = True.")

## 7 · Export to Google Drive (optional)
در صورت نیاز فایل‌های Parquet و گزارش JSON را به Drive کپی کن. محتوای خام شامل داده ویدیویی نیست.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    from pathlib import Path
    import shutil
    drive.mount("/content/drive")
    dest=Path("/content/drive/MyDrive/TajSoccer")
    dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2("outputs/demo_report.json",dest/"demo_report.json")
    print("Saved:",dest)
else:
    print("Local outputs live in /content/Taj_soccer_recommender/outputs")

## Scientific and legal limitations
- IDSSE: tracking + comprehensive event feed, CC BY 4.0.
- SkillCorner: partial broadcast coverage, extrapolated points (`is_detected=False`), dynamic events **not equivalent** to a complete event feed.
- Do not infer a team weakness from a single frame, or interpret heuristic scores as goal probabilities.
- For real scouting, inspect video, derive period-correct directions, and validate on matches held out in time.

See `docs/METHODOLOGY.md` and `README.md` for limitations and next steps.